In [5]:
import pandas as pd
from pathlib import Path

DATA_PATH = Path.cwd().parent / "data" / "cardio_train.csv"
df = pd.read_csv(DATA_PATH, sep=";")

df["age_years"] = (df["age"] / 365.25).round(1)
df["bmi"] = (df["weight"] / (df["height"] / 100) ** 2).round(1)

df.describe().round(1)

,id,age,gender,height,weight,ap_hi,ap_lo,cholesterol,gluc,smoke,alco,active,cardio,age_years,bmi
count,70000.0,70000.0,70000.0,70000.0,70000.0,70000.0,70000.0,70000.0,70000.0,70000.0,70000.0,70000.0,70000.0,70000.0,70000.0
mean,49972.4,19468.9,1.3,164.4,74.2,128.8,96.6,1.4,1.2,0.1,0.1,0.8,0.5,53.3,27.6
std,28851.3,2467.3,0.5,8.2,14.4,154.0,188.5,0.7,0.6,0.3,0.2,0.4,0.5,6.8,6.1
min,0.0,10798.0,1.0,55.0,10.0,-150.0,-70.0,1.0,1.0,0.0,0.0,0.0,0.0,29.6,3.5
25%,25006.8,17664.0,1.0,159.0,65.0,120.0,80.0,1.0,1.0,0.0,0.0,1.0,0.0,48.4,23.9
50%,50001.5,19703.0,1.0,165.0,72.0,120.0,80.0,1.0,1.0,0.0,0.0,1.0,0.0,53.9,26.4
75%,74889.2,21327.0,2.0,170.0,82.0,140.0,90.0,2.0,1.0,0.0,0.0,1.0,1.0,58.4,30.2
max,99999.0,23713.0,2.0,250.0,200.0,16020.0,11000.0,3.0,3.0,1.0,1.0,1.0,1.0,64.9,298.7


In [6]:
print("Heart disease cases:")
print(df["cardio"].value_counts())
print("\nImpossible blood pressure values:")
print("ap_hi out of range:", ((df["ap_hi"] < 80) | (df["ap_hi"] > 250)).sum())
print("ap_lo out of range:", ((df["ap_lo"] < 40) | (df["ap_lo"] > 150)).sum())
print("ap_lo higher than ap_hi:", (df["ap_lo"] > df["ap_hi"]).sum())

Heart disease cases:
cardio
0    35021
1    34979
Name: count, dtype: int64

Impossible blood pressure values:
ap_hi out of range: 247
ap_lo out of range: 1034
ap_lo higher than ap_hi: 1234


In [7]:
before = len(df)

clean = df[
    df["ap_hi"].between(80, 250)
    & df["ap_lo"].between(40, 150)
    & (df["ap_lo"] < df["ap_hi"])
    & df["height"].between(120, 220)
    & df["weight"].between(30, 200)
    & df["bmi"].between(12, 60)
].copy()

print(f"Rows before: {before}")
print(f"Rows after:  {len(clean)}")
print(f"Removed:     {before - len(clean)} ({(before - len(clean)) / before:.1%})")
print("\nStill balanced?")
print(clean["cardio"].value_counts())

CLEAN_PATH = Path.cwd().parent / "data" / "cardio_clean.csv"
clean.to_csv(CLEAN_PATH, index=False)
print(f"\nSaved to {CLEAN_PATH.name}")

Rows before: 70000
Rows after:  68581
Removed:     1419 (2.0%)

Still balanced?
cardio
0    34654
1    33927
Name: count, dtype: int64

Saved to cardio_clean.csv


In [8]:
for col in ["cholesterol", "gluc", "smoke", "active"]:
    rate = clean.groupby(col)["cardio"].mean().mul(100).round(1)
    print(f"{col}: % with heart disease")
    print(rate.to_string(), "\n")

cholesterol: % with heart disease
cholesterol
1    43.5
2    59.6
3    76.3 

gluc: % with heart disease
gluc
1    47.6
2    58.9
3    61.8 

smoke: % with heart disease
smoke
0    49.7
1    46.9 

active: % with heart disease
active
0    53.3
1    48.5 

